1. Data Preprocessing and Exploration
在数据预处理与探索阶段，我们需要加载隆德气象数据集 lund_daily_weather.csv 并检查数据结构，利用描述性统计（Descriptive Statistics）计算气温、气压和湿度等特征的基础统计量，通过绘制直方图（Histograms）展示最高温、露点差与气压波动等特征的分布，利用散点图（Scatter Plots）分析气压与湿度的关联，同时利用 shift(-1) 错开一天构建次日降雨目标变量（Target Alignment），并采用四分位距（IQR）或箱线图等过滤条件识别和剔除异常天气数据（Outliers）。
2. Unsupervised Learning – Clustering
在无监督学习与聚类阶段，我们需要挑选露点差、昼夜温差和气压变幅等核心气象特征（Feature Selection），使用 K-Means 或 PCA 算法将每日天气无监督聚类为不同的天气模式（Weather Patterns，如晴朗干燥型、阴湿高压型与低压风暴型），并采用平滑函数（Smoothing Functions）对气象指标进行平滑处理与特征插值，通过对比平滑处理前后数据的分布直方图来验证聚类特征对天气形态的表达能力与稳定性。
3. Supervised Learning – Classification
在监督学习与分类阶段，我们需要以构建好的次日降雨标签 RainTomorrow 为目标，使用 train_test_split 将数据集划分为训练集和测试集，训练决策树（Decision Tree）和随机森林（Random Forest）作为基准分类模型，利用准确率（Accuracy）、精确率（Precision）、召回率（Recall）、F1-score 以及混淆矩阵（Confusion Matrix）评估模型性能，并通过对物理特征参数进行极值与均值拟合缩放（Optimal Parameter Scaling），深入分析传统模型在气象预测中的优缺点与瓶颈。
4. Neural Network Implementation
在神经网络实现阶段，我们需要使用 PyTorch 搭建适合静态表格分类的多层感知机（MLP）网络，引入 Dropout 层与早停法（Early Stopping）等正则化技术防止过拟合，以交叉熵损失驱动训练并提取高阶特征组合，随后在测试集上使用与监督学习相同的评估指标（Accuracy, F1-score, Confusion Matrix）衡量性能，最后对比神经网络与传统随机森林模型的分类效果、泛化能力以及单次预测耗时（Inference Speed）。
5. Data Visualization and Presentation
在数据可视化与结果汇报阶段，我们需要绘制神经网络在各个训练轮次（Epochs）中的训练与验证损失曲线（Loss Curves）及准确率变化曲线（Accuracy Curves）以验证收敛性，同时创建特征重要性条形图（Feature Importance）与 ROC-AUC 曲线等可视化图表突出核心洞察，最终将完整的数据管线（Data Pipeline）、模型对比结果、代码片段与物理解释整合为一份结构清晰、逻辑完备的终极汇报报告（Final Presentation & Report）。

In [1]:
!pip install joblib
!pip install requests

In [2]:
import pandas as pd
import requests

# ==============================================================================
# 1. Configuration & API Data Fetching
# ==============================================================================
# Location parameters for Lund, Sweden
LATITUDE = 55.71
LONGITUDE = 13.19
START_DATE = "2010-01-01"
END_DATE = "2024-12-31"

# Open-Meteo Historical Archive API Endpoint
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "precipitation",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
    ],
    "timezone": "Europe/Berlin",
}

print("Fetching historical weather observations for Lund from Open-Meteo...")
response = requests.get(url, params=params, timeout=20)
response.raise_for_status()
data = response.json()

# ==============================================================================
# 2. Raw Hourly Data Structuring
# ==============================================================================
df_hourly = pd.DataFrame(data["hourly"])
df_hourly["time"] = pd.to_datetime(df_hourly["time"])
df_hourly["date"] = df_hourly["time"].dt.date

# ==============================================================================
# 3. Daily Static Feature Aggregation (Features for Day n)
# ==============================================================================
print("Aggregating hourly measurements into daily feature representations...")
df_daily = (
    df_hourly.groupby("date")
    .agg(
        temp_max=("temperature_2m", "max"),
        temp_min=("temperature_2m", "min"),
        temp_mean=("temperature_2m", "mean"),
        humidity_mean=("relative_humidity_2m", "mean"),
        dewpoint_mean=("dew_point_2m", "mean"),
        pressure_mean=("surface_pressure", "mean"),
        cloud_cover_mean=("cloud_cover", "mean"),
        wind_speed_max=("wind_speed_10m", "max"),
        precipitation_today=("precipitation", "sum"),
    )
    .reset_index()
)
# ==============================================================================
# 4. Feature Engineering
# ==============================================================================
# Diurnal temperature range (Day n)
df_daily["temp_range"] = df_daily["temp_max"] - df_daily["temp_min"]

# Dewpoint spread: Proxy for atmospheric moisture saturation (Day n)
df_daily["dewpoint_spread"] = df_daily["temp_mean"] - df_daily["dewpoint_mean"]

# Day-over-day barometric pressure trend
df_daily["pressure_change"] = df_daily["pressure_mean"].diff().fillna(0.0)

# Seasonal cycle signal
df_daily["month"] = pd.to_datetime(df_daily["date"]).dt.month

# ==============================================================================
# 5. Target Label Alignment (Label for Day n + 1)
# ==============================================================================
# Shift target labels backward using shift(-1) so that meteorological features
# from Day n are paired with the binary precipitation outcome of Day n + 1.
# RainTomorrow = 1 if Day n+1 precipitation > 0.1 mm, otherwise 0.
df_daily["rain_tomorrow"] = (
    df_daily["precipitation_today"].shift(-1) > 0.1
).astype(int)

# Drop the final row as shift(-1) creates a NaN label for the last day
df_daily = df_daily.iloc[:-1].reset_index(drop=True)

# ==============================================================================
# 6. Save Processed Dataset to Local Storage
# ==============================================================================
output_csv = "lund_daily_weather.csv"
df_daily.to_csv(output_csv, index=False)

print(
    f"\nDataset processing complete! Created {len(df_daily)} daily samples."
)
print(f"Saved dataset to '{output_csv}'.")
print("\nFirst 5 rows of the aligned tabular dataset:")
print(df_daily.head())

Fetching historical weather observations for Lund from Open-Meteo...
Aggregating hourly measurements into daily feature representations...

Dataset processing complete! Created 5478 daily samples.
Saved dataset to 'lund_daily_weather.csv'.

First 5 rows of the aligned tabular dataset:
         date  temp_max  temp_min  temp_mean  humidity_mean  dewpoint_mean  \
0  2010-01-01      -4.7      -8.3  -6.741667      80.208333      -9.629167   
1  2010-01-02      -4.0      -9.5  -6.683333      78.291667      -9.900000   
2  2010-01-03      -5.9     -11.5  -9.350000      75.791667     -12.862500   
3  2010-01-04      -1.0      -6.1  -3.137500      86.541667      -5.079167   
4  2010-01-05      -0.7      -2.6  -1.825000      93.166667      -2.779167   

   pressure_mean  cloud_cover_mean  wind_speed_max  precipitation_today  \
0     996.700000         83.125000            14.1                  0.0   
1    1007.425000         22.875000            19.4                  0.0   
2    1013.662500    